# Build Porseman training dataset

Create one positive answer and seven LLM-reviewed hard negatives for each training question. This notebook reads only the training split; test and validation data remain untouched.

## Setup

Locate the project root so the notebook works when opened from either the repository root or the `notebooks` directory.

In [1]:
import csv
from pathlib import Path

def find_project_root(start_path):
    for candidate in (start_path, *start_path.parents):
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the project root. Run this notebook from inside the repository.")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
TRAIN_INPUT_PATH = PROJECT_ROOT / "data/processed/porseman_train.csv"
OUTPUT_PATH = PROJECT_ROOT / "data/processed/porseman_train_with_hard_negatives.jsonl"

RETRIEVAL_MODEL_NAME = "BAAI/bge-m3"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
RETRIEVAL_CANDIDATE_COUNT = 100
RERANKER_CANDIDATE_COUNT = 20
NEGATIVES_PER_QUERY = 7

print(f"Training input: {TRAIN_INPUT_PATH}")
print(f"Output dataset: {OUTPUT_PATH}")

Training input: /home/rahnema/aahmadi/finetune-embedding-models/data/processed/porseman_train.csv
Output dataset: /home/rahnema/aahmadi/finetune-embedding-models/data/processed/porseman_train_with_hard_negatives.jsonl


## 1. Read the training split

Load the train split and verify the fields required for hard-negative mining. Every row retains its stable ID, question, and positive answer.

In [2]:
REQUIRED_FIELDS = {"id", "question", "content_text"}

if not TRAIN_INPUT_PATH.is_file():
    raise FileNotFoundError(f"Training CSV was not found: {TRAIN_INPUT_PATH}")

with TRAIN_INPUT_PATH.open(encoding="utf-8-sig", newline="") as source:
    reader = csv.DictReader(source)
    missing_fields = REQUIRED_FIELDS - set(reader.fieldnames or [])
    if missing_fields:
        raise ValueError(f"Training CSV is missing fields: {sorted(missing_fields)}")
    train_rows = [
        {field: (row[field] or "").strip() for field in REQUIRED_FIELDS}
        for row in reader
    ]

invalid_rows = [row for row in train_rows if not all(row.values())]
if invalid_rows:
    raise ValueError(f"Training CSV has {len(invalid_rows):,} row(s) with an empty required field.")

train_ids = [row["id"] for row in train_rows]
train_queries = [row["question"] for row in train_rows]
train_positives = [row["content_text"] for row in train_rows]

if len(train_ids) != len(set(train_ids)):
    raise ValueError("Training CSV has duplicate IDs.")

print(f"Training rows: {len(train_rows):,}")
print(f"Unique positive answers: {len(set(train_positives)):,}")

Training rows: 12,807
Unique positive answers: 12,807


In [3]:
## 2. Check the runtime environment

import importlib.util

import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")

print(f"FlagEmbedding installed: {importlib.util.find_spec('FlagEmbedding') is not None}")

PyTorch version: 2.7.0+cu126
CUDA available: True
GPU: NVIDIA GeForce RTX 3090
CUDA version: 12.6
FlagEmbedding installed: True


In [4]:
## 3. Check FlagEmbedding version

from importlib.metadata import version

print(f"FlagEmbedding version: {version('FlagEmbedding')}")

FlagEmbedding version: 1.3.5


In [5]:
## 4. Inspect BGE-M3 API

import inspect

from FlagEmbedding import BGEM3FlagModel

print("BGEM3FlagModel.__init__:")
print(inspect.signature(BGEM3FlagModel.__init__))

print("\nBGEM3FlagModel.encode:")
print(inspect.signature(BGEM3FlagModel.encode))

BGEM3FlagModel.__init__:
(self, model_name_or_path: str, normalize_embeddings: bool = True, use_fp16: bool = True, query_instruction_for_retrieval: Optional[str] = None, query_instruction_format: str = '{}{}', devices: Union[str, List[str], NoneType] = None, pooling_method: str = 'cls', trust_remote_code: bool = False, cache_dir: Optional[str] = None, colbert_dim: int = -1, batch_size: int = 256, query_max_length: int = 512, passage_max_length: int = 512, return_dense: bool = True, return_sparse: bool = False, return_colbert_vecs: bool = False, **kwargs: Any)

BGEM3FlagModel.encode:
(self, sentences: Union[List[str], str], batch_size: Optional[int] = None, max_length: Optional[int] = None, return_dense: Optional[bool] = None, return_sparse: Optional[bool] = None, return_colbert_vecs: Optional[bool] = None, **kwargs: Any) -> Dict[Literal['dense_vecs', 'lexical_weights', 'colbert_vecs'], Union[numpy.ndarray, List[Dict[str, float]], List[numpy.ndarray]]]


In [6]:
## 5. Inspect token-length distribution

import numpy as np

from transformers import AutoTokenizer
from tqdm.auto import tqdm

tokenizer = AutoTokenizer.from_pretrained(RETRIEVAL_MODEL_NAME)


def get_token_lengths(texts, batch_size=256):
    lengths = []

    for start in tqdm(
        range(0, len(texts), batch_size),
        desc="Counting tokens",
    ):
        batch = texts[start:start + batch_size]

        encoded = tokenizer(
            batch,
            add_special_tokens=True,
            truncation=False,
            return_length=True,
        )

        lengths.extend(encoded["length"])

    return np.asarray(lengths)


query_token_lengths = get_token_lengths(train_queries)
answer_token_lengths = get_token_lengths(train_positives)


def print_length_stats(name, lengths):
    print(f"\n{name}")
    print(f"min:    {lengths.min():,}")
    print(f"median: {np.median(lengths):,.0f}")
    print(f"p90:    {np.percentile(lengths, 90):,.0f}")
    print(f"p95:    {np.percentile(lengths, 95):,.0f}")
    print(f"p99:    {np.percentile(lengths, 99):,.0f}")
    print(f"max:    {lengths.max():,}")
    print(f">512:   {(lengths > 512).sum():,}")
    print(f">1024:  {(lengths > 1024).sum():,}")
    print(f">2048:  {(lengths > 2048).sum():,}")
    print(f">4096:  {(lengths > 4096).sum():,}")


print_length_stats("Questions", query_token_lengths)
print_length_stats("Answers", answer_token_lengths)

Counting tokens:   0%|          | 0/51 [00:00<?, ?it/s]

Counting tokens:   0%|          | 0/51 [00:00<?, ?it/s]


Questions
min:    6
median: 24
p90:    60
p95:    82
p99:    156
max:    574
>512:   1
>1024:  0
>2048:  0
>4096:  0

Answers
min:    24
median: 356
p90:    860
p95:    942
p99:    1,007
max:    1,023
>512:   4,497
>1024:  0
>2048:  0
>4096:  0


In [7]:
## 6. Load the dense retrieval model

from FlagEmbedding import BGEM3FlagModel

QUERY_MAX_LENGTH = 1024
PASSAGE_MAX_LENGTH = 1024

retrieval_model = BGEM3FlagModel(
    RETRIEVAL_MODEL_NAME,
    use_fp16=True,
    normalize_embeddings=True,
    devices="cuda",
    query_max_length=QUERY_MAX_LENGTH,
    passage_max_length=PASSAGE_MAX_LENGTH,
    return_dense=True,
    return_sparse=False,
    return_colbert_vecs=False,
)

print("Retrieval model loaded successfully.")
print(f"Query max length: {QUERY_MAX_LENGTH}")
print(f"Passage max length: {PASSAGE_MAX_LENGTH}")

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Retrieval model loaded successfully.
Query max length: 1024
Passage max length: 1024


In [8]:
## 7. Smoke-test BGE-M3 encoding

sample_queries = train_queries[:2]
sample_answers = train_positives[:2]

query_output = retrieval_model.encode(
    sample_queries,
    batch_size=2,
    max_length=QUERY_MAX_LENGTH,
)

answer_output = retrieval_model.encode(
    sample_answers,
    batch_size=2,
    max_length=PASSAGE_MAX_LENGTH,
)

query_embeddings = query_output["dense_vecs"]
answer_embeddings = answer_output["dense_vecs"]

print("Query embeddings:")
print(f"shape: {query_embeddings.shape}")
print(f"dtype: {query_embeddings.dtype}")
print(f"norms: {np.linalg.norm(query_embeddings, axis=1)}")

print("\nAnswer embeddings:")
print(f"shape: {answer_embeddings.shape}")
print(f"dtype: {answer_embeddings.dtype}")
print(f"norms: {np.linalg.norm(answer_embeddings, axis=1)}")

pre tokenize: 100%|████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 1778.00it/s]
You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
Inference Embeddings: 100%|██████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 38.47it/s]

Query embeddings:
shape: (2, 1024)
dtype: float16
norms: [0.9995 1.    ]

Answer embeddings:
shape: (2, 1024)
dtype: float16
norms: [1. 1.]


In [9]:
## 8. Stress-test answer encoding on the longest samples

longest_answer_indices = np.argsort(answer_token_lengths)[-16:]
longest_answers = [train_positives[i] for i in longest_answer_indices]

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

test_output = retrieval_model.encode(
    longest_answers,
    batch_size=16,
    max_length=PASSAGE_MAX_LENGTH,
)

test_embeddings = test_output["dense_vecs"]

peak_memory_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

print(f"Embeddings shape: {test_embeddings.shape}")
print(f"Longest token length: {answer_token_lengths[longest_answer_indices].max()}")
print(f"Peak GPU memory allocated: {peak_memory_gb:.2f} GB")

Inference Embeddings: 100%|██████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.49it/s]

Embeddings shape: (16, 1024)
Longest token length: 1023
Peak GPU memory allocated: 1.44 GB


In [10]:
## 9. Test corpus encoding batch size

test_answers = train_positives[:64]

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

test_output = retrieval_model.encode(
    test_answers,
    batch_size=64,
    max_length=PASSAGE_MAX_LENGTH,
)

test_embeddings = test_output["dense_vecs"]

peak_memory_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)

print(f"Embeddings shape: {test_embeddings.shape}")
print(f"Peak GPU memory allocated: {peak_memory_gb:.2f} GB")

Inference Embeddings: 100%|██████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.50s/it]

Embeddings shape: (64, 1024)
Peak GPU memory allocated: 2.56 GB


In [12]:
## 10. Encode the answer corpus

CORPUS_BATCH_SIZE = 64

torch.cuda.empty_cache()

corpus_output = retrieval_model.encode(
    train_positives,
    batch_size=CORPUS_BATCH_SIZE,
    max_length=PASSAGE_MAX_LENGTH,
)

corpus_embeddings = corpus_output["dense_vecs"]

print(f"Shape: {corpus_embeddings.shape}")
print(f"Dtype: {corpus_embeddings.dtype}")
print(
    f"Average norm: "
    f"{np.mean(np.linalg.norm(corpus_embeddings, axis=1)):.4f}"
)

Inference Embeddings: 100%|██████████████████████████████████████████████████████████████| 201/201 [01:02<00:00,  3.24it/s]


Shape: (12807, 1024)
Dtype: float16
Average norm: 1.0000


In [13]:
## 11. Encode the query corpus

QUERY_BATCH_SIZE = 64

torch.cuda.empty_cache()

query_output = retrieval_model.encode(
    train_queries,
    batch_size=QUERY_BATCH_SIZE,
    max_length=QUERY_MAX_LENGTH,
)

query_embeddings = query_output["dense_vecs"]

print(f"Shape: {query_embeddings.shape}")
print(f"Dtype: {query_embeddings.dtype}")
print(
    f"Average norm: "
    f"{np.mean(np.linalg.norm(query_embeddings, axis=1)):.4f}"
)

Inference Embeddings: 100%|██████████████████████████████████████████████████████████████| 201/201 [00:05<00:00, 35.32it/s]


Shape: (12807, 1024)
Dtype: float16
Average norm: 1.0000


In [14]:
## 12. Retrieve top-100 answers for each query

import numpy as np
from tqdm.auto import tqdm

TOP_K = 100

def retrieve_top_k(
    query_embeddings,
    corpus_embeddings,
    top_k=100,
    batch_size=128,
):
    all_indices = []
    all_scores = []

    for start in tqdm(
        range(0, len(query_embeddings), batch_size),
        desc="Retrieving",
    ):
        batch_queries = query_embeddings[start:start + batch_size]

        scores = batch_queries @ corpus_embeddings.T

        top_indices = np.argpartition(
            -scores,
            kth=top_k - 1,
            axis=1,
        )[:, :top_k]

        top_scores = np.take_along_axis(
            scores,
            top_indices,
            axis=1,
        )

        order = np.argsort(
            -top_scores,
            axis=1,
        )

        top_indices = np.take_along_axis(
            top_indices,
            order,
            axis=1,
        )

        top_scores = np.take_along_axis(
            top_scores,
            order,
            axis=1,
        )

        all_indices.append(top_indices)
        all_scores.append(top_scores)

    return (
        np.vstack(all_indices),
        np.vstack(all_scores),
    )


retrieval_indices, retrieval_scores = retrieve_top_k(
    query_embeddings,
    corpus_embeddings,
    top_k=TOP_K,
)

print(f"Indices shape: {retrieval_indices.shape}")
print(f"Scores shape: {retrieval_scores.shape}")

Retrieving:   0%|          | 0/101 [00:00<?, ?it/s]

Indices shape: (12807, 100)
Scores shape: (12807, 100)


In [15]:
## 12. Retrieve top-100 answers (GPU)

import torch
from tqdm.auto import tqdm

TOP_K = 100

query_tensor = torch.from_numpy(query_embeddings).cuda()
corpus_tensor = torch.from_numpy(corpus_embeddings).cuda()

retrieval_indices = []
retrieval_scores = []

with torch.no_grad():
    for start in tqdm(
        range(0, len(query_tensor), 512),
        desc="GPU Retrieval",
    ):
        batch_queries = query_tensor[start:start + 512]

        scores = batch_queries @ corpus_tensor.T

        top_scores, top_indices = torch.topk(
            scores,
            k=TOP_K,
            dim=1,
        )

        retrieval_indices.append(
            top_indices.cpu()
        )

        retrieval_scores.append(
            top_scores.cpu()
        )

retrieval_indices = torch.cat(
    retrieval_indices,
    dim=0,
).numpy()

retrieval_scores = torch.cat(
    retrieval_scores,
    dim=0,
).numpy()


print(f"Indices shape: {retrieval_indices.shape}")
print(f"Scores shape: {retrieval_scores.shape}")

GPU Retrieval:   0%|          | 0/26 [00:00<?, ?it/s]

Indices shape: (12807, 100)
Scores shape: (12807, 100)


In [16]:
## 13. Inspect retrieval quality

sample_index = 0

print("QUESTION:")
print(train_queries[sample_index])

print("\nTRUE POSITIVE:")
print(train_positives[sample_index])

print("\nTOP-10 RETRIEVED:")
for rank, (idx, score) in enumerate(
    zip(
        retrieval_indices[sample_index][:10],
        retrieval_scores[sample_index][:10],
    ),
    start=1,
):
    print("\n" + "-" * 80)
    print(f"Rank: {rank}")
    print(f"Score: {score:.4f}")
    print(f"ID: {train_ids[idx]}")
    print(train_positives[idx][:500])

QUESTION:
روزه‌هایى که در اوایل سن تکلیف به جا نیاورده‌ام، علاوه بر قضا کفّاره هم دارد؟

TRUE POSITIVE:
همه مراجع: هر مقدار از روزه‌ها را که نگرفته‌اید، باید قضا کنید و افزون بر آن، براى هر روز نیز باید کفّاره بدهید؛ یعنى، دو ماه روزه بگیرید یا شصت فقیر را سیر کنید و یا به هر کدام یک مد (تقریبا ده سیر) طعام (گندم یا جو و مانند آن) به آنها بدهید. [ توضیح‌المسائل مراجع، م 1660؛ وحید، توضیح‌المسائل، م 1668.]

TOP-10 RETRIEVED:

--------------------------------------------------------------------------------
Rank: 1
Score: 0.7563
ID: porseman-12610
همه مراجع: خیر، تنها قضاى روزه‌ها واجب است و کفّاره ندارد؛ ولى اگر قضاى روزه‌ها را تا ماه رمضان سال بعد به تأخیر اندازد، به جهت تأخیر، باید براى هر روز، یک مد طعام کفّاره بدهد.[ توضیح‌المسائل مراجع، م 1705؛ وحید، توضیح‌المسائل، م1713.]

--------------------------------------------------------------------------------
Rank: 2
Score: 0.7256
ID: porseman-12473
همه مراجع (به جز بهجت، فاضل و مکارم): خیر، فعلاً مکلف به کفاره نیستند، بلکه تکلیف شرعى آنه

In [17]:
## 14. Remove positive and exact duplicate candidates

def normalize_for_exact_match(text):
    return " ".join(text.split())


normalized_positives = [
    normalize_for_exact_match(x)
    for x in train_positives
]


def filter_candidates(query_index):
    query_id = train_ids[query_index]
    positive_text = normalized_positives[query_index]

    filtered = []

    for rank, (candidate_idx, score) in enumerate(
        zip(
            retrieval_indices[query_index],
            retrieval_scores[query_index],
        ),
        start=1,
    ):
        candidate_idx = int(candidate_idx)

        # Remove the same record
        if train_ids[candidate_idx] == query_id:
            continue

        # Remove exact duplicate answer
        if normalized_positives[candidate_idx] == positive_text:
            continue

        filtered.append(
            {
                "index": candidate_idx,
                "rank": rank,
                "score": float(score),
                "answer": train_positives[candidate_idx],
            }
        )

    return filtered


sample_filtered = filter_candidates(0)

print(f"Candidates before filtering: {TOP_K}")
print(f"Candidates after filtering: {len(sample_filtered)}")

print("\nFirst 10 filtered candidates:")
for item in sample_filtered[:10]:
    print("-" * 80)
    print(f"rank={item['rank']} score={item['score']:.4f}")
    print(item["answer"][:300])

Candidates before filtering: 100
Candidates after filtering: 99

First 10 filtered candidates:
--------------------------------------------------------------------------------
rank=1 score=0.7563
همه مراجع: خیر، تنها قضاى روزه‌ها واجب است و کفّاره ندارد؛ ولى اگر قضاى روزه‌ها را تا ماه رمضان سال بعد به تأخیر اندازد، به جهت تأخیر، باید براى هر روز، یک مد طعام کفّاره بدهد.[ توضیح‌المسائل مراجع، م 1705؛ وحید، توضیح‌المسائل، م1713.]
--------------------------------------------------------------------------------
rank=2 score=0.7256
همه مراجع (به جز بهجت، فاضل و مکارم): خیر، فعلاً مکلف به کفاره نیستند، بلکه تکلیف شرعى آنها این است که تا رمضان سال بعد روزه‌ها را قضا کنند و اگر تا آن موقع قضا نکردند و تأخیر انداختند، مکلف به پرداخت کفاره تأخیر مى‌شوند، لذا اگر قبل از آن پرداخت کنند حساب نمى‌شود و باید مجدداً بدهند. [ فاضل، استفتا
--------------------------------------------------------------------------------
rank=3 score=0.7021
همه مراجع (به جز سیستانى، خامنه‌اى و مکارم): در فرض یاد شده، باید

In [18]:
## 15. Load BGE reranker

from FlagEmbedding import FlagReranker

reranker = FlagReranker(
    RERANKER_MODEL_NAME,
    use_fp16=True,
    devices="cuda:1",
)

print("Reranker loaded successfully.")

Reranker loaded successfully.


In [19]:
## 16. Test reranking on one query

sample_query_index = 0

candidates = sample_filtered[:10]

pairs = [
    (
        train_queries[sample_query_index],
        item["answer"],
    )
    for item in candidates
]

rerank_scores = reranker.compute_score(
    pairs,
    normalize=True,
)

for item, score in zip(candidates, rerank_scores):
    print("-" * 80)
    print(f"Retrieval rank: {item['rank']}")
    print(f"Retrieval score: {item['score']:.4f}")
    print(f"Reranker score: {score:.4f}")
    print(item["answer"][:300])

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


--------------------------------------------------------------------------------
Retrieval rank: 1
Retrieval score: 0.7563
Reranker score: 0.8968
همه مراجع: خیر، تنها قضاى روزه‌ها واجب است و کفّاره ندارد؛ ولى اگر قضاى روزه‌ها را تا ماه رمضان سال بعد به تأخیر اندازد، به جهت تأخیر، باید براى هر روز، یک مد طعام کفّاره بدهد.[ توضیح‌المسائل مراجع، م 1705؛ وحید، توضیح‌المسائل، م1713.]
--------------------------------------------------------------------------------
Retrieval rank: 2
Retrieval score: 0.7256
Reranker score: 0.7833
همه مراجع (به جز بهجت، فاضل و مکارم): خیر، فعلاً مکلف به کفاره نیستند، بلکه تکلیف شرعى آنها این است که تا رمضان سال بعد روزه‌ها را قضا کنند و اگر تا آن موقع قضا نکردند و تأخیر انداختند، مکلف به پرداخت کفاره تأخیر مى‌شوند، لذا اگر قبل از آن پرداخت کنند حساب نمى‌شود و باید مجدداً بدهند. [ فاضل، استفتا
--------------------------------------------------------------------------------
Retrieval rank: 3
Retrieval score: 0.7021
Reranker score: 0.7275
همه مراجع (به جز سیستانى،

In [20]:
## 17. Benchmark reranking batch size

RERANK_TOP_K = 20
RERANK_BATCH_SIZE = 32

sample_size = 32

test_pairs = []

for i in range(sample_size):
    candidates = filter_candidates(i)[:100]

    for item in candidates:
        test_pairs.append(
            (
                train_queries[i],
                item["answer"],
            )
        )

print(f"Pairs to rerank: {len(test_pairs)}")

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(device=1)

scores = reranker.compute_score(
    test_pairs,
    normalize=True,
    batch_size=RERANK_BATCH_SIZE,
)

peak_memory_gb = (
    torch.cuda.max_memory_allocated(device=1)
    / (1024 ** 3)
)

print(f"Scores returned: {len(scores)}")
print(f"Peak GPU memory: {peak_memory_gb:.2f} GB")

Pairs to rerank: 3171


Compute Scores: 100%|████████████████████████████████████████████████████████████████████| 100/100 [00:13<00:00,  7.47it/s]

Scores returned: 3171
Peak GPU memory: 1.43 GB


In [ ]:
## 18. Rerank all retrieved candidates

RERANK_BATCH_SIZE = 32
RERANKED_CANDIDATE_COUNT = 20

all_reranked_candidates = []

torch.cuda.empty_cache()

for query_index in tqdm(
    range(len(train_rows)),
    desc="Reranking queries",
):
    candidates = filter_candidates(query_index)

    pairs = [
        (
            train_queries[query_index],
            item["answer"],
        )
        for item in candidates
    ]

    scores = reranker.compute_score(
        pairs,
        normalize=True,
        batch_size=RERANK_BATCH_SIZE,
    )

    for item, score in zip(candidates, scores):
        item["reranker_score"] = float(score)

    candidates = sorted(
        candidates,
        key=lambda x: x["reranker_score"],
        reverse=True,
    )

    all_reranked_candidates.append(
        candidates[:RERANKED_CANDIDATE_COUNT]
    )

print(
    f"Queries reranked: "
    f"{len(all_reranked_candidates)}"
)

print(
    f"Candidates for first query: "
    f"{len(all_reranked_candidates[0])}"
)

In [22]:
## 18. Build reranker pairs

rerank_pairs = []
rerank_metadata = []

for query_index in tqdm(
    range(len(train_rows)),
    desc="Building pairs",
):
    candidates = filter_candidates(query_index)

    for item in candidates:
        rerank_pairs.append(
            (
                train_queries[query_index],
                item["answer"],
            )
        )

        rerank_metadata.append(
            {
                "query_index": query_index,
                "candidate_index": item["index"],
                "retrieval_rank": item["rank"],
                "retrieval_score": item["score"],
            }
        )

print(f"Total pairs: {len(rerank_pairs):,}")
print(f"Metadata entries: {len(rerank_metadata):,}")

Building pairs:   0%|          | 0/12807 [00:00<?, ?it/s]

Total pairs: 1,268,529
Metadata entries: 1,268,529


In [23]:
## 19. Batch rerank all pairs

import numpy as np
from tqdm.auto import tqdm

RERANK_CHUNK_SIZE = 5000
RERANK_BATCH_SIZE = 64

all_reranker_scores = []

torch.cuda.empty_cache()

for start in tqdm(
    range(0, len(rerank_pairs), RERANK_CHUNK_SIZE),
    desc="Batch reranking",
):
    chunk = rerank_pairs[start:start + RERANK_CHUNK_SIZE]

    scores = reranker.compute_score(
        chunk,
        normalize=True,
        batch_size=RERANK_BATCH_SIZE,
    )

    all_reranker_scores.extend(scores)


all_reranker_scores = np.asarray(
    all_reranker_scores,
    dtype=np.float32,
)

print(f"Scores shape: {all_reranker_scores.shape}")
print(f"Min score: {all_reranker_scores.min():.4f}")
print(f"Max score: {all_reranker_scores.max():.4f}")


Compute Scores: 100%|██████████████████████████████████████████████████████████████████████| 79/79 [00:21<00:00,  3.62it/s]

pre tokenize: 100%|████████████████████████████████████████████████████████████████████████| 79/79 [00:01<00:00, 70.20it/s]

Compute Scores: 100%|██████████████████████████████████████████████████████████████████████| 79/79 [00:20<00:00,  3.80it/s]

pre tokenize: 100%|████████████████████████████████████████████████████████████████████████| 79/79 [00:00<00:00, 85.12it/s]

Compute Scores: 100%|██████████████████████████████████████████████████████████████████████| 79/79 [00:21<00:00,  3.72it/s]

pre tokenize: 100%|████████████████████████████████████████████████████████████████████████| 56/56 [00:00<00:00, 87.59it/s]

Compute Scores: 100%|██████████████████████████████████████████████████████████████████████| 56/56 [00:15<00:00,  3.65it/s]

Scores shape: (1268529,)
Min score: 0.0000
Max score: 1.0000


In [24]:
## 20. Attach reranker scores and keep top-20

RERANKED_CANDIDATE_COUNT = 20

for metadata, score in zip(
    rerank_metadata,
    all_reranker_scores,
):
    metadata["reranker_score"] = float(score)


query_to_candidates = {}

for item in rerank_metadata:
    query_index = item["query_index"]

    query_to_candidates.setdefault(
        query_index,
        []
    ).append(item)


top20_candidates = {}

for query_index, candidates in query_to_candidates.items():
    candidates = sorted(
        candidates,
        key=lambda x: x["reranker_score"],
        reverse=True,
    )

    top20_candidates[query_index] = candidates[:RERANKED_CANDIDATE_COUNT]


print(f"Queries: {len(top20_candidates)}")
print(
    f"Candidates for query 0: "
    f"{len(top20_candidates[0])}"
)

Queries: 12807
Candidates for query 0: 20


In [25]:
## 21. Build LLM review records (preview)

llm_review_records = []

for query_index in range(len(train_rows)):
    candidates = []

    for item in top20_candidates[query_index]:
        candidate_index = item["candidate_index"]

        candidates.append(
            {
                "candidate_id": train_ids[candidate_index],
                "candidate_answer": train_positives[candidate_index],
                "retrieval_rank": item["retrieval_rank"],
                "retrieval_score": item["retrieval_score"],
                "reranker_score": item["reranker_score"],
            }
        )

    llm_review_records.append(
        {
            "id": train_ids[query_index],
            "query": train_queries[query_index],
            "positive": train_positives[query_index],
            "candidates": candidates,
        }
    )

print(f"Records: {len(llm_review_records)}")
print(f"Candidates in first record: {len(llm_review_records[0]['candidates'])}")

Records: 12807
Candidates in first record: 20


In [26]:
## 22. Validate LLM review records

errors = []

for record in llm_review_records[:100]:
    for candidate in record["candidates"]:
        if candidate["candidate_id"] == record["id"]:
            errors.append(
                {
                    "query_id": record["id"],
                    "candidate_id": candidate["candidate_id"],
                }
            )

print(f"Self-match errors in first 100 records: {len(errors)}")

if errors:
    print(errors[:5])

Self-match errors in first 100 records: 0


In [27]:
## 23. Verify candidate IDs can be resolved from dataset IDs

all_record_ids = set(train_ids)

missing_candidate_ids = []

for record in llm_review_records:
    for candidate in record["candidates"]:
        candidate_id = candidate["candidate_id"]

        if candidate_id not in all_record_ids:
            missing_candidate_ids.append(candidate_id)

print(f"Total records: {len(llm_review_records):,}")
print(f"Known train IDs: {len(all_record_ids):,}")
print(f"Missing candidate IDs: {len(missing_candidate_ids):,}")

if missing_candidate_ids:
    print("Examples:")
    print(missing_candidate_ids[:10])

Total records: 12,807
Known train IDs: 12,807
Missing candidate IDs: 0


In [28]:
## 24. Build compact LLM review dataset

compact_llm_records = []

for record in llm_review_records:
    compact_candidates = []

    for candidate in record["candidates"]:
        compact_candidates.append(
            {
                "candidate_id": candidate["candidate_id"],
                "retrieval_rank": candidate["retrieval_rank"],
                "retrieval_score": candidate["retrieval_score"],
                "reranker_score": candidate["reranker_score"],
            }
        )

    compact_llm_records.append(
        {
            "id": record["id"],
            "query": record["query"],
            "positive": record["positive"],
            "candidates": compact_candidates,
        }
    )

print(f"Records: {len(compact_llm_records):,}")
print(
    f"Candidates in first record: "
    f"{len(compact_llm_records[0]['candidates'])}"
)

Records: 12,807
Candidates in first record: 20


In [29]:
## 25. Inspect compact format

import json

print(
    json.dumps(
        compact_llm_records[0],
        ensure_ascii=False,
        indent=2,
    )[:2000]
)

{
  "id": "porseman-12457",
  "query": "روزه‌هایى که در اوایل سن تکلیف به جا نیاورده‌ام، علاوه بر قضا کفّاره هم دارد؟",
  "positive": "همه مراجع: هر مقدار از روزه‌ها را که نگرفته‌اید، باید قضا کنید و افزون بر آن، براى هر روز نیز باید کفّاره بدهید؛ یعنى، دو ماه روزه بگیرید یا شصت فقیر را سیر کنید و یا به هر کدام یک مد (تقریبا ده سیر) طعام (گندم یا جو و مانند آن) به آنها بدهید. [ توضیح‌المسائل مراجع، م 1660؛ وحید، توضیح‌المسائل، م 1668.]",
  "candidates": [
    {
      "candidate_id": "porseman-34656",
      "retrieval_rank": 13,
      "retrieval_score": 0.68994140625,
      "reranker_score": 0.9742394685745239
    },
    {
      "candidate_id": "porseman-12610",
      "retrieval_rank": 1,
      "retrieval_score": 0.75634765625,
      "reranker_score": 0.8966140151023865
    },
    {
      "candidate_id": "porseman-35427",
      "retrieval_rank": 15,
      "retrieval_score": 0.68505859375,
      "reranker_score": 0.8958876132965088
    },
    {
      "candidate_id": "porseman-12542",
   

In [30]:
## 26. Final validation before export

errors = {
    "wrong_candidate_count": 0,
    "self_candidate": 0,
    "candidate_text_leak": 0,
}

for record in compact_llm_records:
    if len(record["candidates"]) != 20:
        errors["wrong_candidate_count"] += 1

    for candidate in record["candidates"]:
        if candidate["candidate_id"] == record["id"]:
            errors["self_candidate"] += 1

        if "candidate_answer" in candidate:
            errors["candidate_text_leak"] += 1


print(errors)

{'wrong_candidate_count': 0, 'self_candidate': 0, 'candidate_text_leak': 0}


In [31]:
## 27. Validate against original train data

train_by_id = {
    row["id"]: row
    for row in train_rows
}

validation_errors = {
    "missing_record_id": 0,
    "query_mismatch": 0,
    "positive_mismatch": 0,
    "missing_candidate_id": 0,
}

for record in compact_llm_records:
    source = train_by_id.get(record["id"])

    if source is None:
        validation_errors["missing_record_id"] += 1
        continue

    if record["query"] != source["question"]:
        validation_errors["query_mismatch"] += 1

    if record["positive"] != source["content_text"]:
        validation_errors["positive_mismatch"] += 1

    for candidate in record["candidates"]:
        if candidate["candidate_id"] not in train_by_id:
            validation_errors["missing_candidate_id"] += 1


print(validation_errors)

{'missing_record_id': 0, 'query_mismatch': 0, 'positive_mismatch': 0, 'missing_candidate_id': 0}


In [32]:
## 28. Export compact LLM review dataset

import json

LLM_OUTPUT_PATH = PROJECT_ROOT / "data/processed/porseman_llm_review_compact.jsonl"

with LLM_OUTPUT_PATH.open(
    "w",
    encoding="utf-8",
) as f:
    for record in compact_llm_records:
        f.write(
            json.dumps(
                record,
                ensure_ascii=False,
            )
            + "\n"
        )

print(f"Saved: {LLM_OUTPUT_PATH}")
print(
    f"Size: "
    f"{LLM_OUTPUT_PATH.stat().st_size / (1024**2):.2f} MB"
)

Saved: /home/rahnema/aahmadi/finetune-embedding-models/data/processed/porseman_llm_review_compact.jsonl
Size: 64.02 MB
